Notebook per la creazione e l'analisi del dataset scaricato da PDB: PL- v.2020R1

1. estrazione informazioni da INDEX_general_PL.2020R1.ist
2. creazione csv
3. salvataggio file corretti pdb per proteina e sdf per ligand
4. analisi e visualizzazione
5. 




In [25]:
import os
import re #regex library
import csv
import pandas as pd

#ignore warinings
import warnings
warnings.filterwarnings("ignore")

## **Analisi e Preprocessing Dataset**


### **Estrazione informazioni INDEX**

In [26]:
# read and extract info from INDEX_general_PL
datapath = "data\\PDB"
filename = "INDEX_general_PL.2020R1.lst"
filepath = os.path.join(datapath, filename)

if not os.path.exists(filepath):
    raise FileNotFoundError(f"{filename} file not found path[{filepath}]")

#read file
with open(filepath, "r", encoding="utf-8") as f:
    lines = [
        ln.strip()
        for ln in f
        if ln.strip() and not ln.lstrip().startswith("#")
    ]
    
#regex [PDB_code, resolution, release_year, binding_data[type,compare,unit], reference, ligand_name, comments]
regex= re.compile(r"(\w+)\s+"+                              #pdb code
                  r"([\d.]+|NMR)\s+"+                       #resolution
                  r"(\d{4})\s+"+                            #release year
                  r"(\w+)([=<~]{0,2})([\d\.]+)(\w+)\s+"     #binding data
                  r"//\s[\w\.]+\s+"                         #file path reference (ignored)
                  r"\(([^)]+?)\)"                           #ligand name
                  r"\s*(.*)"                                #comments
                  )

parsed_index = []
skipped = 0
for ln in lines:
    match = regex.match(ln)
    if match:
        (pdb, res, year, btype, comp, val, unit, file, comment) = match.groups()
        
        parsed_index.append({
            "PDB_code": pdb,
            "resolution": res,
            "release_year": year,
            "binding_type": btype,
            "compare": comp,
            "value": val,
            "unit": unit,
            "comments": comment.strip()
        })
    else:
        skipped += 1
        #print("SKIPPED:", ln)
print(f"Parsed: {len(parsed_index)} | Skipped: {skipped}")

#salvo csv
csv_filename = "parsed_index.csv"
csv_filepath = os.path.join(datapath, csv_filename)

with open(csv_filepath, "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=parsed_index[0].keys())
    writer.writeheader()
    writer.writerows(parsed_index)
       


Parsed: 19037 | Skipped: 0


In [27]:
df = pd.DataFrame(parsed_index)
df.head()

,PDB_code,resolution,release_year,binding_type,compare,value,unit,comments
0,2tpi,2.10,1982,Kd,=,49,uM,
1,5tln,2.30,1982,Ki,=,0.43,uM,incomplete ligand structure
2,4tln,2.30,1982,Ki,=,190,uM,
3,4cts,2.90,1984,Kd,<,10,uM,
4,6rsa,NMR,1986,Ki,=,40,uM,
